# Project · Regression from scratch on real medical data

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app). Predict disease progression from 10 patient measurements (scikit-learn's diabetes dataset, 442 patients, bundled with Colab). You'll build linear regression four ways (closed form, gradient descent, ridge with cross-validation) and put an honest error bar on one coefficient with the bootstrap. Uses Murphy ch. 4, 5, 7, 8 and 11.

Work through the milestones in order; each builds on the previous cell's variables. Solutions are hidden: double-click a solution cell's title to reveal it.

## Milestone 1 · Load, split and standardise

Load the data, split 80/20 with a fixed seed, and standardise each feature using the training set's mean and standard deviation only. Why must the test set use the training statistics?

In [ ]:
# Your turn
import numpy as np
from sklearn.datasets import load_diabetes
d = load_diabetes()
X, y, names = d.data, d.target, d.feature_names
rng = np.random.default_rng(0)
idx = rng.permutation(len(y)); cut = int(0.8 * len(y))
tr, te = idx[:cut], idx[cut:]
mu, sd = X[tr].mean(0), X[tr].std(0)
Xtr = ...  # TODO: standardise with TRAINING statistics
Xte = ...  # TODO: same transform for the test set
ytr, yte = y[tr], y[te]
print(Xtr.shape, Xte.shape, names)

In [ ]:
# @title Solution 1 (double-click to show the code)
import numpy as np
from sklearn.datasets import load_diabetes
d = load_diabetes()
X, y, names = d.data, d.target, d.feature_names
rng = np.random.default_rng(0)
idx = rng.permutation(len(y)); cut = int(0.8 * len(y))
tr, te = idx[:cut], idx[cut:]
mu, sd = X[tr].mean(0), X[tr].std(0)
Xtr = (X[tr] - mu) / sd  # standardise with TRAINING statistics
Xte = (X[te] - mu) / sd  # same transform for the test set
ytr, yte = y[tr], y[te]
print(Xtr.shape, Xte.shape, names)

<details><summary><b>Answer and explanation</b></summary>

Standardising puts every feature on the same scale, so weights are comparable and gradient descent isn't dominated by one large-scale feature. Using test-set statistics would leak information about the test data into training, making the test score optimistic.

Expected output:
```
(353, 10) (89, 10) ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
```

</details>

## Milestone 2 · Closed-form least squares

Add a column of ones for the bias and solve the normal equations. Report the test MSE and R² (1 − MSE / variance of y). Which feature has the largest weight?

In [ ]:
# Your turn
A = np.column_stack([np.ones(len(Xtr)), Xtr])
w = ...  # TODO: normal equations (X^T X) w = X^T y
pred = np.column_stack([np.ones(len(Xte)), Xte]) @ w
mse = np.mean((pred - yte) ** 2)
print(f"test MSE {mse:.1f}   R^2 {1 - mse / yte.var():.3f}")
print("largest weight:", names[np.argmax(np.abs(w[1:]))], np.round(w[1:], 1))

In [ ]:
# @title Solution 2 (double-click to show the code)
A = np.column_stack([np.ones(len(Xtr)), Xtr])
w = np.linalg.solve(A.T @ A, A.T @ ytr)  # normal equations (X^T X) w = X^T y
pred = np.column_stack([np.ones(len(Xte)), Xte]) @ w
mse = np.mean((pred - yte) ** 2)
print(f"test MSE {mse:.1f}   R^2 {1 - mse / yte.var():.3f}")
print("largest weight:", names[np.argmax(np.abs(w[1:]))], np.round(w[1:], 1))

<details><summary><b>Answer and explanation</b></summary>

The normal equations give the exact minimum of the squared error in one step (Murphy §11.2). Test R² ≈ 0.35 on this split: the 10 measurements explain about a third of the variation in disease progression (other splits give up to ≈ 0.5). BMI and s5 get the largest positive weights; s1 gets a large negative one that partly cancels its correlated twin s2. Because features are standardised, weight sizes are comparable.

Expected output:
```
test MSE 3250.5   R^2 0.346
largest weight: bmi [ -3.4 -11.   29.2  14.  -18.2   6.5  -4.1   8.   28.3   3. ]
```

</details>

## Milestone 3 · The same answer by gradient descent

Minimise the mean squared error with plain gradient descent (learning rate 0.1, 2,000 steps) starting from zeros. Compare the weights with the closed form.

In [ ]:
# Your turn
wg = np.zeros(A.shape[1])
for step in range(2000):
    grad = ...  # TODO: gradient of the mean squared error
    wg -= 0.1 * grad
print("max |w_gd - w_closed| =", np.abs(wg - w).max().round(4))

In [ ]:
# @title Solution 3 (double-click to show the code)
wg = np.zeros(A.shape[1])
for step in range(2000):
    grad = 2 * A.T @ (A @ wg - ytr) / len(ytr)  # gradient of the mean squared error
    wg -= 0.1 * grad
print("max |w_gd - w_closed| =", np.abs(wg - w).max().round(4))

<details><summary><b>Answer and explanation</b></summary>

Most weights match, but the largest gap is still ≈ 0.7 after 2,000 steps. The culprit is s1 and s2: they are strongly correlated, which turns the bowl into a long, flat valley (an ill-conditioned problem, Murphy §8.2.3), and gradient descent crawls along it. The loss is convex, so it will get there; run 20,000 steps and watch the gap shrink. This slow-valley effect is exactly what momentum and Adam fix.

Expected output:
```
max |w_gd - w_closed| = 0.719
```

</details>

## Milestone 4 · Ridge with 5-fold cross-validation

For λ in a log grid from 1e−3 to 1e3, fit ridge (don't penalise the bias) and measure 5-fold CV error on the training set. Pick the best λ, refit on all training data, report test MSE.

In [ ]:
# Your turn
def ridge(A, y, lam):
    P = lam * np.eye(A.shape[1]); P[0, 0] = 0            # don't penalise the bias
    return ...  # TODO: ridge closed form
folds = np.array_split(np.random.default_rng(1).permutation(len(ytr)), 5)
lams = np.logspace(-3, 3, 13); cv = []
for lam in lams:
    errs = []
    for k in range(5):
        v = folds[k]; t = np.concatenate([folds[j] for j in range(5) if j != k])
        wr = ridge(A[t], ytr[t], lam)
        errs.append(np.mean((A[v] @ wr - ytr[v]) ** 2))
    cv.append(np.mean(errs))
best = lams[int(np.argmin(cv))]
wr = ridge(A, ytr, best)
print("CV errors:", np.round(cv, 0))
print(f"best lambda {best:.3g}  test MSE {np.mean((np.column_stack([np.ones(len(Xte)), Xte]) @ wr - yte)**2):.1f}")

In [ ]:
# @title Solution 4 (double-click to show the code)
def ridge(A, y, lam):
    P = lam * np.eye(A.shape[1]); P[0, 0] = 0            # don't penalise the bias
    return np.linalg.solve(A.T @ A + P, A.T @ y)  # ridge closed form
folds = np.array_split(np.random.default_rng(1).permutation(len(ytr)), 5)
lams = np.logspace(-3, 3, 13); cv = []
for lam in lams:
    errs = []
    for k in range(5):
        v = folds[k]; t = np.concatenate([folds[j] for j in range(5) if j != k])
        wr = ridge(A[t], ytr[t], lam)
        errs.append(np.mean((A[v] @ wr - ytr[v]) ** 2))
    cv.append(np.mean(errs))
best = lams[int(np.argmin(cv))]
wr = ridge(A, ytr, best)
print("CV errors:", np.round(cv, 0))
print(f"best lambda {best:.3g}  test MSE {np.mean((np.column_stack([np.ones(len(Xte)), Xte]) @ wr - yte)**2):.1f}")

<details><summary><b>Answer and explanation</b></summary>

Small λ ≈ ordinary least squares; huge λ shrinks every weight to zero and underfits. The CV curve is fairly flat for small λ here because 10 features and 353 patients isn't a hard overfitting problem, and the best λ gives a test MSE close to plain least squares. Ridge helps most when features are many or correlated (s1 and s2 are highly correlated here).

Expected output:
```
CV errors: [3031. 3031. 3031. 3031. 3030. 3028. 3022. 3012. 2998. 2991. 3065. 3411.
 4166.]
best lambda 31.6  test MSE 3234.1
```

</details>

## Milestone 5 · Bootstrap error bar on the BMI effect

Resample the training patients with replacement 2,000 times, refit least squares each time, and report a 95% interval for the BMI weight. Does the interval include 0?

In [ ]:
# Your turn
j = names.index("bmi") + 1
boots = []
rng = np.random.default_rng(2)
for _ in range(2000):
    i = ...  # TODO: resample patients with replacement
    boots.append(np.linalg.lstsq(A[i], ytr[i], rcond=None)[0][j])
lo, hi = np.percentile(boots, [2.5, 97.5])
print(f"BMI weight {w[j]:.1f}   95% bootstrap interval [{lo:.1f}, {hi:.1f}]")

In [ ]:
# @title Solution 5 (double-click to show the code)
j = names.index("bmi") + 1
boots = []
rng = np.random.default_rng(2)
for _ in range(2000):
    i = rng.integers(0, len(ytr), len(ytr))  # resample patients with replacement
    boots.append(np.linalg.lstsq(A[i], ytr[i], rcond=None)[0][j])
lo, hi = np.percentile(boots, [2.5, 97.5])
print(f"BMI weight {w[j]:.1f}   95% bootstrap interval [{lo:.1f}, {hi:.1f}]")

<details><summary><b>Answer and explanation</b></summary>

The interval is well away from 0, so BMI's effect isn't a fluke of this particular sample. The bootstrap gives this error bar without any formula, by rerunning the analysis on resampled data (Murphy §4.7.3).

Expected output:
```
BMI weight 29.2   95% bootstrap interval [22.0, 35.8]
```

</details>

## Milestone 6 · Write-up

In four sentences: what did each method add, and which model would you deploy?

✍️ *Write your answer here.*

<details><summary><b>Model answer</b></summary>

Closed form gives the exact least-squares answer instantly; gradient descent reaches the same answer by a method that also works when no formula exists; ridge plus cross-validation chose how much to shrink using only training data; the bootstrap measured how much a coefficient would wobble with new patients. Deploy ridge with the CV-chosen λ (here nearly identical to least squares), report its test MSE, and report the bootstrap interval for any coefficient you interpret.

</details>